# Source audit: follow a stop to its evidence

The tour metadata records a source and visual link for each stop. This notebook checks that links are present and displays a small source ledger; it does **not** fetch live pages or certify that source contents are unchanged. For publication details and media rights, use [REFERENCES.md](https://github.com/jltobias/JupyterLite-Ebola-1976-2026/blob/main/REFERENCES.md) and the [map catalog](https://github.com/jltobias/JupyterLite-Ebola-1976-2026/blob/main/MAP_CATALOG.md).

In [ ]:
import json
from pathlib import Path

stops = json.loads(Path('../data/tour-stops.json').read_text(encoding='utf-8'))
assert len(stops) == 15
print(f'{len(stops)} curated tour stops, {min(s["year"] for s in stops)}–{max(s["year"] for s in stops)}')

In [ ]:
required = ['id', 'year', 'virus', 'lat', 'lon', 'source', 'sourceUrl', 'visual', 'visualUrl']
missing = [(s.get('id', '?'), key) for s in stops for key in required if key not in s]
non_https = [(s['id'], key) for s in stops for key in ('sourceUrl', 'visualUrl')
             if not s[key].startswith('https://')]
assert not missing, missing
assert not non_https, non_https
assert len({s['id'] for s in stops}) == len(stops)
print('All 15 stops have unique IDs and required, HTTPS source and visual links.')

In [ ]:
from html import escape
from urllib.parse import urlparse
from IPython.display import HTML, display

start_year, end_year = 1976, 2026
selected = [s for s in stops if start_year <= s['year'] <= end_year]
rows = []
for s in selected:
    domain = urlparse(s['sourceUrl']).netloc
    rows.append('<tr>' + ''.join(f'<td>{escape(str(value))}</td>' for value in
                (s['year'], s['title'], s['virus'], domain)) + '</tr>')
table = ('<table><caption>Selected tour stops and source domains; not case counts</caption>'
         '<thead><tr><th>Year</th><th>Stop</th><th>Virus</th><th>Source domain</th></tr></thead>'
         '<tbody>' + ''.join(rows) + '</tbody></table>')
display(HTML(table))

In [ ]:
for s in selected:
    print(f"{s['year']} · {s['title']}\n  Evidence: {s['source']}\n  {s['sourceUrl']}\n  Visual: {s['visual']}\n  {s['visualUrl']}\n")

**Reading the ledger:** A link to a figure does not confer image reuse rights. The 2026 stop cites [WHO's 20 November report](https://www.who.int/emergencies/disease-outbreak-news/item/2026-DON630) for provisional outbreak facts, while an [ERCC map dated 4 June](https://commons.wikimedia.org/wiki/File:ECDM_20260604_DRC_Ebola_outbreak(1).pdf) is explicitly an early visual snapshot. The [Wikipedia outbreak list](https://en.wikipedia.org/w/index.php?title=List_of_Ebola_outbreaks&oldid=1385469746) can help find omissions but is not imported as a surveillance table.